# Exploratory Data Analysis, Preprocessing & Tuning - Credit Scoring Model

This notebook guides you through dataset loading, Exploratory Data Analysis (EDA), pipeline preprocessing, baseline model training, and hyperparameter optimization for the **German Credit Dataset**.

## Part 1: Initial Ingestion & Dataset Audit

### 1.1 Environment Setup & Data Loading

We import pandas, configure local path references, and load the raw German Credit CSV using the project's custom `DataLoader` module.

In [ ]:
import os
import sys
import pandas as pd

# Add root task directory to path to enable package import
sys.path.append(os.path.abspath('..'))
from src.config import Config
from src.data_loader import DataLoader

loader = DataLoader(raw_data_dir='../data/raw')
df = loader.load_dataset()
df.head()

### 1.2 Dataset Shape and Datatype Inspection

We verify the dataset size (1000 rows by 10 columns) and inspect the column datatypes.

In [ ]:
print(f"Row count: {df.shape[0]}")
print(f"Column count: {df.shape[1]}")
print("\nDataset schema and datatypes:")
df.info()

### 1.3 Missing Value Analysis

We identify feature columns containing null values and calculate their percentage of the total sample count.

In [ ]:
missing_counts = df.isnull().sum()
missing_percentage = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentage
})
missing_df[missing_df['Missing Count'] > 0]

### 1.4 Risk Class Balance Check

We check the distribution of the target column (`Risk`) to evaluate imbalance issues.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Risk Class Counts:")
print(df['Risk'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x='Risk', data=df, palette='muted')
plt.title('Credit Risk Target Class Distribution')
plt.xlabel('Risk Status')
plt.ylabel('Count')
plt.show()

## Part 2: Feature Engineering & Preprocessing Pipelines

### 2.1 Custom Feature Engineering

We evaluate custom columns using `FeatureEngineer`:
1. `credit_per_month` = `Credit amount` / `Duration`
2. `age_group` = `young` (<=25), `adult` (<=40), `middle_age` (<=60), `senior` (>60)
3. `credit_to_age_ratio` = `Credit amount` / `Age`

In [ ]:
from src.feature_engineering import FeatureEngineer

fe = FeatureEngineer()
df_engineered = fe.transform(df)
print("Columns after Feature Engineering:", list(df_engineered.columns))
df_engineered[['Age', 'age_group', 'Credit amount', 'Duration', 'credit_per_month', 'credit_to_age_ratio']].head()

### 2.2 Numerical Feature Correlation Analysis

We examine numerical feature correlation patterns after feature engineering to identify collinear relationships.

In [ ]:
numeric_features = ['Age', 'Credit amount', 'Duration', 'credit_per_month', 'credit_to_age_ratio']
correlation_matrix = df_engineered[numeric_features].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.3f', square=True)
plt.title('Numeric Features Correlation Matrix')
plt.show()

### 2.3 Running the Preprocessing Pipeline

We instantiate `Preprocessor` to build the full `ColumnTransformer` pipeline. This transforms raw tabular values into a scaled, imputed, and encoded numeric matrix ready for training models.

In [ ]:
from src.preprocessing import Preprocessor, stratified_train_test_split

# Apply stratified splitting
X_train, X_test, y_train, y_test = stratified_train_test_split(df)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

# Run preprocessor
preprocessor = Preprocessor()
X_train_trans = preprocessor.fit_transform(X_train, y_train)
X_test_trans = preprocessor.transform(X_test)

print(f"Transformed Train shape: {X_train_trans.shape}")
print(f"Transformed Test shape: {X_test_trans.shape}")

## Part 3: Baseline Classifier Model Evaluation & Metrics Guide

### 3.1 Quantitative Evaluation Metrics Guide

When evaluating credit scoring models on an imbalanced dataset (e.g. 70% good, 30% bad risk), simple **Accuracy** can be highly misleading:
1. **Accuracy:** Fraction of correct predictions. If a model predicts 'good' for everyone, it achieves 70% accuracy but fails to catch any defaults.
2. **Precision:** Out of all applicants flagged as high-risk, what fraction actually default? High precision prevents denying credit to solvent applicants.
3. **Recall (Sensitivity):** Out of all actual defaults, what fraction did the model successfully catch? High recall minimizes financial default write-offs.
4. **F1-Score:** The harmonic mean of Precision and Recall, representing a balanced metric of both.
5. **ROC-AUC (Area Under the ROC Curve):** Measures the model's ability to rank high-risk applicants above low-risk applicants across all threshold choices, capturing general classification power.

### 3.2 Why Compare Multiple Models?
Comparing diverse classification families (Logistic Regression, Decision Trees, and Random Forests) helps identify the optimal model. Linear models (Logistic Regression) perform well under linear boundaries, while ensemble architectures (Random Forests) excel at capturing non-linear feature interactions and ignoring noise.

### 3.3 Load & Display Baseline Leaderboard Results

Let's load the comparison table generated during the pipeline execution and visualize the metrics.

In [ ]:
# Load model comparison stats
comp_df = pd.read_csv('../results/metrics/model_comparison.csv')

# Melt for visualization
df_melted = comp_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

# Plot comparisons
plt.figure(figsize=(9, 5))
sns.barplot(x='Metric', y='Score', hue='Model', data=df_melted, palette='Set2')
plt.title('Credit Risk Classifiers - Quantitative Metrics Comparison')
plt.xlabel('Evaluation Metric')
plt.ylabel('Score Value')
plt.ylim(0, 1.05)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

comp_df

## Part 4: Hyperparameter Optimization & Final Model Selection

### 4.1 Hyperparameter Tuning Concepts

1. **What is Hyperparameter Tuning?**
   Unlike standard weights (which the algorithm learns during training), hyperparameters are configurations set by the engineer *before* training starts (e.g., the depth of a tree, or the number of estimators). Tuning adjusts these limits to find the optimal trade-off between model simplicity (avoiding overfitting) and fit capacity (avoiding underfitting).
   
2. **Why use RandomizedSearchCV?**
   `RandomizedSearchCV` samples a fixed number of parameter combinations from a designated distribution grid rather than running an exhaustive search (`GridSearchCV`). This achieves virtually identical score improvements while consuming a fraction of the memory and processing power, making it perfect for laptop runs.
   
3. **Why is ROC-AUC the Target Metric?**
   ROC-AUC evaluates performance across all decision probability thresholds. In credit scoring, adjusting decision boundaries later in production allows banks to slide thresholds to match current risk tolerances. Optimizing for ROC-AUC preserves high threshold-agnostic discriminative power.
   
4. **How does Final Model Selection work?**
   We test model configurations against held-out validation folds using cross-validation to protect against leakage. We then select the candidate with the highest ROC-AUC on the independent test split, confirming it has generalized well.

### 4.2 Load & Display Tuning Results

Let's load the best parameters and final report to evaluate our optimization gains.

In [ ]:
# Print best parameters discovered
print("Best Hyperparameters:")
with open('../results/metrics/best_parameters.txt', 'r') as f:
    print(f.read())

# Print baseline vs optimized final performance comparison
print("\nPerformance Report:")
with open('../results/metrics/final_model_report.txt', 'r') as f:
    print(f.read())

### 4.3 Feature Importance of Final Production Model

We can inspect the relative contributions of the features as calculated by Gini importance inside our optimized Random Forest classifier.

In [ ]:
import joblib

# Load final model checkpoint
checkpoint = joblib.load('../models/final_credit_scoring_model.joblib')
best_rf = checkpoint['model']
fitted_prep = checkpoint['preprocessor']

feature_names = fitted_prep.get_feature_names()
importances = best_rf.feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=feat_imp.values[:15], y=feat_imp.index[:15], palette='viridis')
plt.title('Optimized Random Forest Feature Importances (Top 15)')
plt.xlabel('Gini Importance Score')
plt.ylabel('Feature Name')
plt.show()